In [1]:
import os
import sys
from collections import defaultdict
import pandas as pd

module_path = os.path.abspath(os.path.join('..'))
if module_path not in sys.path: sys.path.append(module_path)
from ytmusic_library import YTMusicPlaylists

RUN_API_AUTH_TEST = True
HEADER_FILE = '../oauth.json'
PLAYLIST_TSV_DIR = '../playlists/'

Y = YTMusicPlaylists(header=HEADER_FILE, playlist_tsv_dir=PLAYLIST_TSV_DIR)
if RUN_API_AUTH_TEST: Y.test_ytmusic_api()
print(f"Loaded {len(Y.playlists['title'].unique())} playlists")

Using header file: ../oauth.json
Test Passed in 3.77 seconds
Using ytmusicapi version: 1.3.1
Loaded 464 playlists


# Process Each Playlist

##### TODO add playlist to super playliusts if exist see pdf
##### TODO auto gnerate some date based like playlsiysd
##### TODO move based on playcount (if not LIKE infer NOT_LIKE based on large playcount)
##### TODO make into script, run monthly



In [2]:
# Skip playlistes inferred as these kind
do_dry_run = False  # make sure all NOT OK is fine

verbose = False  # Verbose printing

# Options for cleaning radio/indifferent playlists
move_like=True
min_num_like=10
sleep=1
create_like_playlist=True
remove_not_like=True
remove_dislike=True

# Options for checking inferred playlist kind
playlist_skip_kinds = ('SKIP', 'ALBUM', 'YT_GENERATED')
skip_if_dislike=True
like_playlist_min_like_pct = 80
not_like_playlist_max_like_pct = 20
radio_playlist_max_like_pct = 50

# Options for processing playlists
duplicate_threshold = 5  # was 4 first run
playlist_skip_starts_with = ['zz not like']


like_results = {}
radio_results = {}
pl_ct = defaultdict(int)
playlists_kinds = {k: set() for k in Y._valid_playlist_kinds}
n_playlists = len(Y.playlists)
for i, p in Y.playlists.iterrows():
    if (i+1) % 100 == 0:
        print(f'{i+1} of {n_playlists} playlists processed')
    pl_ct['playlists_processed'] += 1
    for skip_str in playlist_skip_starts_with:
        if p.title.startswith(skip_str):
            pl_ct[f'playlist_skip_starts_with_{skip_str}'] += 1
            print(f' SKIPPING playlist: {p.title} starts with {skip_str}')
            continue

    if verbose:
        print(f"Playlist: {p.title} ({p.playlistId}) has {p['count']} tracks")

    # Infer playlist kind from the title, default to LIKE if nothing inferred
    pl_kind = Y.infer_playlist_kind(p)

    # Maybe skip playlist right away
    # Skip when unable to infer kind (ideally this is not called)
    if not pl_kind:
        pl_ct['playlist_kind_not_inferred'] += 1
        print(f'SKIPPING playlist: {p.title} can not infer kind', 
              f'for playlist need manual fix (add to _like_playlists.tsv?)')
        continue
    pl_ct[f'playlist_kind_is_{pl_kind.lower()}'] += 1

    # Decide to skip playlist based on playlist kind
    playlists_kinds[pl_kind].add(p.title)
    if pl_kind in playlist_skip_kinds:
        pl_ct['playlist_kind_in_playlist_skip_list'] += 1
        if verbose:
            print(f'SKIPPING playlist: {p.title} as it is',
                f'a kind flagged for skipping: {pl_kind}')
        continue

    # Query playlist tracks and other metadata
    p_info = Y.playlist_get_info(p.playlistId, playlist_limit=Y.playlist_limit)
    if p_info['trackCount'] == 0:
        pl_ct['playlist_is_empty'] += 1
        print(f'SKIPPING playlist: {p.title} No tracks in playlist')
        continue
    # Check max length of playlist
    if len(p_info['tracks']) >= Y.playlist_limit:
        pl_ct[f'playlist_has_>{Y.playlist_limit}_limit'] += 1
        print(f'SKIPPING playlist: {p.title} which has',
              f'{Y.playlist_limit} or more tracks ({len(p_info["tracks"])})')
        continue
    # Check playlist privacy
    if p_info['privacy'] == 'PUBLIC':
        pl_ct['playlist_is_public'] += 1
        print(f'SKIPPING playlist: {p.title} which has',
              f'privacy: {p_info["privacy"]}')
        continue
    elif p_info['privacy'] == 'UNLISTED':
        pl_ct['playlist_privacy_is_unlisted'] += 1
        if verbose:
            print(f' playlist: {p.title} has privacy {p_info["privacy"]}')

    # Get ratings for playlist tracks
    ratings = {k: set() for k in Y._valid_ratings}
    for track in p_info["tracks"]:
        pl_ct['track_processed'] += 1
        if track["likeStatus"] not in ratings.keys():
            pl_ct['track_rating_is_none'] += 1
            ratings['NONE'].add(track["videoId"])
        else:
            pl_ct['track_rating_exists'] += 1
            ratings[track["likeStatus"]].add(track["videoId"])

    # See if playlist is correctly flagged as LIKE or RADIO
    like_percent = round(100*len(ratings["LIKE"])/len(p_info["tracks"]))
    if not Y._is_playlist_kind_ok(pl_kind, like_percent,
                                  like_playlist_min_like_pct, 
                                  not_like_playlist_max_like_pct,
                                  radio_playlist_max_like_pct):
        pl_ct['playlist_kind_not_ok'] += 1
        print(f'WARNING NOT OK playlist: {p.title} of kind {pl_kind}',
                f'({like_percent}% liked) has: {len(ratings["LIKE"])} likes,',
                f'{len(ratings["DISLIKE"])} dislikes,{len(ratings["INDIFFERENT"])}',
                f'indifferent, {len(ratings["NONE"])} none')


    # Potentially alter playlist, or generate new playlists
    if do_dry_run:
        continue
    # Remove duplicates from playlist
    new_pl_id = Y.playlist_remove_duplicates(p_info, duplicate_threshold, verbose=verbose)
    if p.playlistId != new_pl_id:
        p.playlistId = new_pl_id
        pl_ct['playlist_removed_duplicates'] += 1
        p_info = Y.playlist_get_info(
            new_pl_id, playlist_limit=Y.playlist_limit, use_cache=False)
    # Like all tracks in playlist if kind is LIKE
    if pl_kind == 'LIKE':
        like_results[p.title] = Y.playlist_rate_all_songs(
            p_info, rating=pl_kind, skip_if_dislike=skip_if_dislike, verbose=verbose)
        continue
    # Split radio playlist into LIKE vs RADIO
    elif pl_kind == 'INDIFFERENT':
        radio_results[p.title] = Y.clean_up_radio_playlist(
            pl_info=Y.playlist_get_info(p.playlistId, use_cache=True),
            verbose=verbose, sleep=sleep,
            move_like=move_like, min_num_like=min_num_like, 
            create_like_playlist=create_like_playlist,
            remove_dislike=remove_dislike,  remove_not_like=remove_not_like)
        continue



Playlist ambiant electro: Rated 2 of 65 tracks as LIKE
Playlist ambient: Rated 4 of 757 tracks as LIKE
Playlist ambient Indie Synths: Rated 2 of 84 tracks as LIKE
Playlist beats: Rated 38 of 1109 tracks as LIKE
Playlist beats 2010s wonky LA scene: Rated 15 of 419 tracks as LIKE
Playlist beats instrumental: 5 duplicate tracks removed (408 of 413 unique)
Playlist beats instrumental: Rated 11 of 408 tracks as LIKE
Playlist Beats Lofi: Rated 1 of 75 tracks as LIKE
Playlist beats Soulful Instrumentals: Rated 6 of 213 tracks as LIKE
Playlist beats trap dj: Rated 4 of 69 tracks as LIKE
Playlist beats_chill dj: Rated 2 of 132 tracks as LIKE
Playlist beats_jazzy dj: Rated 1 of 56 tracks as LIKE
Playlist beats_phat dj: Rated 2 of 61 tracks as LIKE
Playlist beats_raw dj: Rated 3 of 114 tracks as LIKE
Playlist beats_soul dj: Rated 5 of 99 tracks as LIKE
Playlist beats_wonky dj: Rated 1 of 86 tracks as LIKE
Playlist blues: 6 duplicate tracks removed (727 of 733 unique)
Playlist blues: Rated 7 of 72

In [17]:
like_results_ = pd.DataFrame(like_results).T.sort_values('track_rated', ascending=False)

radio_results_ = pd.DataFrame(radio_results).T.sort_values('moved_like')

,removed_dislike,moved_like,removed_not_like,like_and_not_like
ambient modern radio,0,0,0,0
x_r.90sPunk_tracks_radio,0,0,0,0
x_r.witchHouse_tracks_radio,0,0,0,0
x_r.80sMusic_tracks_radio,0,0,0,0
synth trip radio,0,0,0,0
...,...,...,...,...
brass radio,0,9,0,1
x_r.DoomMetal_tracks_radio,0,9,0,0
Japanese City Pop 70s 80s radio,0,10,0,0
electronic Innerwaves radio,0,10,0,0


In [23]:
pl_ct = pd.DataFrame.from_dict(pl_ct, orient='index')
pl_ct.columns = ['count']
print(f'Final Playlist Cleanup Counters:\n{pl_ct}')

like_results = pd.DataFrame(like_results).T.sort_values('track_rated', ascending=False)

radio_results = pd.DataFrame(radio_results).T.sort_values('moved_like')
radio_results['total_changes'] = radio_results.sum(axis=1)
radio_results = radio_results.sort_values('total_changes', ascending=False)

Final Playlist Cleanup Counters:
                                        count
playlist_is_public                          1
playlist_kind_in_playlist_skip_list        73
playlist_kind_is_album                     58
playlist_kind_is_indifferent              140
playlist_kind_is_like                     252
playlist_kind_is_skip                      10
playlist_kind_is_yt_generated               5
playlist_privacy_is_unlisted               43
playlist_removed_duplicates                13
playlist_skip_starts_with_zz not like       2
playlists_processed                       465
track_processed                        116522
track_rating_exists                    115695
track_rating_is_none                      827


In [24]:
_like_cleanup_file = '../playlists/_ytmusic_cleanup_like_playlists_results.tsv'
_radio_cleanup_file = '../playlists/_ytmusic_cleanup_radio_playlists_results.tsv'
_cleanup_counters_file = '../playlists/_ytmusic_cleanup_playlist_counters.tsv'


like_results.to_csv(_like_cleanup_file, sep='\t')
radio_results.to_csv(_radio_cleanup_file, sep='\t')
pl_ct.to_csv(_cleanup_counters_file, sep='\t')

